
# 创建项目目录 
- 项目根目录 名称： quant-mlops-project
- 创建子目录  mkdir -p src/data data/raw notebooks
- 注意：以下记录内容，如果没有特别说明，所有命令都是在根目录下执行

# 使用uv作为python虚拟环境
- 安装   sudo UV_INSTALL_DIR=/usr/local/bin sh -c 'curl -LsSf https://astral.sh/uv/install.sh | sh'
- 查看版本信息  uv --version

# 配置uv环境
- uv init --python 3.11

# 安装python 到当前uv环境的，uv 管理的所有 Python 都在 ~/.local/share/uv/python/ 下，一个版本一个目录。这是 uv 的私有仓库。
- uv python install 3.11  
# 验证结果，并生成.venv文件
- uv run python --version

# 激活uv 环境
- source .venv/bin/activate

# 安装依赖库
- uv add jupyter
- uv add yfinance pandas pyarrow 


# 在 uv 项目里给 Jupyter 加 kernel
- uv add ipykernel 
- uv run python -m ipykernel install --user --name=quant-mlops --display-name="Python (quant-mlops)"

# 数据下载
0. 写 src/data/tickers.py
- 详见 quant-mlops-project/src/data/tickers.py   一共有100只大盘股
2. 写 src/data/download.py
- 具体见：quant-mlops-project/src/data/download.py
3.执行数据下载 uv run python src/data/download.py

In [5]:
# 检查数据情况
! uv run python -c "import pandas as pd; df = pd.read_parquet('../data/raw/AAPL.parquet'); print(df.head()); print(df.dtypes); print(df.shape)"

        date      close       high        low       open     volume ticker
0 2015-01-02  24.171757  24.638257  23.733998  24.627201  212818400   AAPL
1 2015-01-05  23.490795  24.021411  23.305080  23.941819  257142000   AAPL
2 2015-01-06  23.493011  23.751686  23.132634  23.554916  263188400   AAPL
3 2015-01-07  23.822439  23.921929  23.590294  23.700839  160423600   AAPL
4 2015-01-08  24.737745  24.795229  24.032468  24.149647  237458000   AAPL
date      datetime64[ms]
close            float64
high             float64
low              float64
open             float64
volume             int64
ticker               str
dtype: object
(2952, 7)


# 数据校验 + 处理层
- 目标：把 100 个 raw parquet 合成一张干净的、可校验的、带元数据的表。这一步决定后面特征能不能做对。

1. 写 src/data/validate.py
- 详见：quant-mlops-project/src/data/validate.py


In [17]:
import os
print(os.getcwd())

/home/mlops/quant-mlops-project/notebooks


In [20]:
# 跑验证数据的脚本：notebook中的脚本
import subprocess

result = subprocess.run(
    ["uv", "run", "python", "/home/mlops/quant-mlops-project/src/data/validate.py"],
    cwd="/home/mlops/quant-mlops-project",
    capture_output=True, text=True
)

print("returncode:", result.returncode)
print("stdout:")
print(result.stdout)
print("stderr:")
print(result.stderr)

returncode: 0
stdout:
[OK] AAPL: 2952 rows, 2015-01-02 ~ 2026-09-29
[OK] AMZN: 2952 rows, 2015-01-02 ~ 2026-09-29
[OK] GOOGL: 2952 rows, 2015-01-02 ~ 2026-09-29
[OK] JPM: 2952 rows, 2015-01-02 ~ 2026-09-29
[OK] META: 2952 rows, 2015-01-02 ~ 2026-09-29
[OK] MSFT: 2952 rows, 2015-01-02 ~ 2026-09-29
[OK] NVDA: 2952 rows, 2015-01-02 ~ 2026-09-29
[OK] TSLA: 2952 rows, 2015-01-02 ~ 2026-09-29
[OK] UNH: 2952 rows, 2015-01-02 ~ 2026-09-29
[OK] V: 2952 rows, 2015-01-02 ~ 2026-09-29

stderr:



In [ ]:
# 在终端中测试的命令：
uv run python src/data/validate.py

2. 写 src/data/build_processed.py，连接10张表，并做数据质量校验
- 详见  quant-mlops-project/src/data/build_processed.py
- 在终端中，执行脚本：

(quant-mlops-project) mlops@MS-CEVXSRKPSHOI:~/quant-mlops-project$ uv run python src/data/build_processed.py
saved 29520 rows -> data/processed/prices.parquet
date range: 2015-01-02 ~ 2026-09-29
tickers: 10

In [23]:
import pandas as pd
df = pd.read_parquet('/home/mlops/quant-mlops-project/data/processed/prices.parquet')
df.describe()

,date,open,high,low,close,volume
count,29520,29520.000000,29520.000000,29520.000000,29520.000000,2.952000e+04
mean,2020-11-12 19:25:51.219000,166.276524,168.245443,164.277405,166.306379,8.246939e+07
min,2015-01-02 00:00:00,0.462494,0.466802,0.453397,0.458185,7.140000e+05
25%,2017-12-05 18:00:00,54.346125,55.025655,53.733708,54.381028,1.174888e+07
50%,2020-11-10 12:00:00,134.431684,135.870138,132.823037,134.444244,3.002665e+07
75%,2023-10-18 06:00:00,231.267836,233.826462,228.637505,231.529903,8.012445e+07
max,2026-09-29 00:00:00,787.943661,793.022965,777.655507,786.798218,3.692928e+09
std,NaN,139.340862,140.993209,137.636396,139.323878,1.491981e+08


In [24]:
df.head()

,date,ticker,open,high,low,close,volume
0,2015-01-02,AAPL,24.627201,24.638257,23.733998,24.171757,212818400
1,2015-01-02,AMZN,15.629000,15.737500,15.348000,15.426000,55664000
2,2015-01-02,GOOGL,26.378983,26.537477,26.145208,26.227922,26480000
3,2015-01-02,JPM,45.610710,46.182860,45.530021,45.838104,12600000
4,2015-01-02,META,77.834529,78.181207,76.962872,77.705757,18177500


In [25]:
df.tail()

,date,ticker,open,high,low,close,volume
29515,2026-09-29,MSFT,508.309998,513.640015,502.399994,508.959991,21341800
29516,2026-09-29,NVDA,230.970001,232.820007,227.029999,227.210007,101494900
29517,2026-09-29,TSLA,358.399994,358.739990,351.670013,352.839996,32492400
29518,2026-09-29,UNH,374.880005,376.589996,371.329987,374.940002,3284700
29519,2026-09-29,V,366.929993,368.260010,365.000000,365.880005,5471000
